In [8]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os, math
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import importlib, core
importlib.reload(core)
from core import (count_sites_per_sample_ptm_report, process_ptm_site_report,
                  calculate_dilution_linearity, _hex_to_rgba)

In [9]:
# Shape-number gradient (light->dark across 100..1500 shapes)
SHAPE_ORDER = [100, 200, 300, 500, 1000, 1500]
SHAPE_COLORS = ['#EEA69B', '#E78373', '#E1604C', '#DB452E', '#B3321E', '#641C11']
SITE_COLORS  = ['#ef745c', '#d06257', '#b15052', '#923e4d', '#722b47', '#34073d']
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# Data upload

In [10]:
# phosphoDVP shape-number dilution series: laser-microdissected mouse-brain "shapes"
# (100-1500), wide PTM Site Reports (Class I per-run), n=4 replicates each.
RAW_DIR = Path('pride_data/analysis_data/revision/figure5')
shapes = {n: pd.read_csv(next(RAW_DIR.glob(f'*DVPP_{n}shapes_Report.tsv')), sep='\t', low_memory=False)
          for n in SHAPE_ORDER}
print('shape datasets:', list(shapes))

shape datasets: [100, 200, 300, 500, 1000, 1500]


# Figure 5b
Class I phosphosite depth as a function of number of microdissected shapes (n=4 per shape number). Strict per-run Class I, multiplicity collapsed, localization enforced.

In [11]:
# Figure 5b - Class I phosphosite depth vs shape number (box + points, n=4).
counts = {n: list(count_sites_per_sample_ptm_report(shapes[n]).values()) for n in SHAPE_ORDER}
summary = pd.DataFrame({'mean': [int(np.mean(counts[n])) for n in SHAPE_ORDER],
                        'median': [int(np.median(counts[n])) for n in SHAPE_ORDER],
                        'CV%': [round(100*np.std(counts[n], ddof=1)/np.mean(counts[n]), 1) for n in SHAPE_ORDER]},
                       index=SHAPE_ORDER)
summary.index.name = 'shapes'
print(summary.to_string())

fig = go.Figure()
for i, n in enumerate(SHAPE_ORDER):
    ys = counts[n]
    fig.add_trace(go.Box(y=ys, x=[str(n)]*len(ys), name=str(n), boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=8, color=SHAPE_COLORS[i], line=dict(width=0.5, color='black')),
        line=dict(color=SHAPE_COLORS[i], width=1.5), fillcolor=_hex_to_rgba(SHAPE_COLORS[i], 0.2), showlegend=False))
fig.update_layout(width=700, height=600, template='plotly_white',
                  xaxis_title='Number of shapes', yaxis_title='Class I phosphosites')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in SHAPE_ORDER])
fig.update_yaxes(rangemode='tozero')
fig.show()
fig.write_image(r'figures/figure5/figure5b.pdf', width=600, height=600)

        mean  median   CV%
shapes                    
100      244     236  13.2
200      670     677   4.5
300      937     935   3.6
500     1545    1625  16.7
1000    2565    2556   1.9
1500    3617    3663   9.3


# Figure 5c
Per-site dilution linearity across the shape-number series (x = number of shapes). Per project policy, linearity KEEPS multiplicity (per-feature quantitative response); localization enforced. Median R² of the per-site linear fits.

In [12]:
# Figure 5c - per-site linearity vs shape number (keep multiplicity, enforce Class I).
corr_df = calculate_dilution_linearity(shapes, dilution_values=SHAPE_ORDER,
                                       min_dilutions=4, collapse_multiplicity=False, enforce_cutoff=0.75)
a = corr_df['r_squared'].dropna().values
print(f'n sites fit = {len(a):,}   median R^2 = {np.nanmedian(a):.4f}   %(R^2>=0.8) = {100*(a>=0.8).mean():.1f}%')

fig = go.Figure()
fig.add_trace(go.Histogram(x=a, nbinsx=max(len(a)//7, 1),
                           marker=dict(color='black', line=dict(color='black', width=0.3)), opacity=0.8))
fig.add_vline(x=np.nanmedian(a), line={'dash': 'dash', 'width': 3, 'color': 'darkred'})
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='R squared', yaxis_title='Count', showlegend=False, bargap=0.05)
fig.show()
fig.write_image(r'figures/figure5/figure5c.pdf', width=600, height=600)

n sites fit = 1,243   median R^2 = 0.9390   %(R^2>=0.8) = 80.9%


# Figure 5d
Quantitative response of selected well-characterized neuronal phosphosites across the shape-number series (Class I intensities, n=4 per shape). Wording moderated per reviewer: these sites scale with input but the response is **not strictly monotonic at every step** (e.g. Map2 S1783, Prkcg T514) — see the log–log linearity panel in Suppl. Fig. 4 for a quantitative view.

In [13]:
# Figure 5d - selected neuronal phosphosites vs shape number (4 panels in one row).
from plotly.subplots import make_subplots
SITES = ['P20357~Map2_S1783_M1', 'P10637~Mapt_S494_M1', 'P28652~Camk2b_T287_M1', 'P63318~Prkcg_T514_M1']
site_data = {n: process_ptm_site_report(shapes[n], cutoff=0.75)['site_data'] for n in SHAPE_ORDER}

def site_values(key, n):
    sd = site_data[n]
    row = sd[sd['PTM_Collapse_key'] == key]
    if row.empty:
        return []
    cols = [c for c in sd.columns if c not in PROC_META]
    return [v for v in row[cols].iloc[0].values if pd.notna(v)]

labels = [k.split('~')[1] for k in SITES]
fig = make_subplots(rows=1, cols=len(SITES), subplot_titles=labels, horizontal_spacing=0.05)
for j, key in enumerate(SITES):
    for i, n in enumerate(SHAPE_ORDER):
        ys = site_values(key, n)
        fig.add_trace(go.Box(y=ys, x=[str(n)]*len(ys), name=str(n), boxpoints='all', pointpos=0,
            fillcolor=_hex_to_rgba(SHAPE_COLORS[i], 0.5), line=dict(color=SHAPE_COLORS[i]),
            marker=dict(color='black', size=6), showlegend=False), row=1, col=j+1)
    fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in SHAPE_ORDER], row=1, col=j+1)
fig.update_yaxes(title_text='log2 intensity', row=1, col=1)
fig.update_layout(width=2600, height=420, template='plotly_white',
                  margin=dict(b=60))
fig.show()
fig.write_image(r'figures/figure5/figure5d.pdf', width= 2600, height=420)

Dropped 31 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 1,002 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 406 → 401.
Final: 401 sites × 4 samples.
Dropped 261 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 2,789 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,120 → 1,112.
Final: 1,112 sites × 4 samples.
Dropped 232 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 3,910 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,542 → 1,539.
Final: 1,539 sites × 4 samples.
Dropped 926 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75

In [14]:
# Neuron cell-type (5e/5f) helper functions (ported from v00) + reused inputs.
import analytics_core_V04 as ac
from scipy import stats
from scipy.spatial.distance import pdist, squareform
WHISPER = next(RAW_DIR.glob('*Whisper80_Report.tsv'))
PHOS_COND   = r'data/pDVP_neurons_conditionSetup.tsv'
PROT_REPORT = r'pride_data/analysis_data/figure5/DVP_neurons_all.tsv'
PROT_COND   = r'data/DVP_neurons_conditionSetup.tsv'
GROUP4 = ['act_cortex', 'act_subcortex', 'inh_cortex', 'inh_subcortex']

def assign_condition_setup(collapsed_data, condition_df, key_col='PTM_Collapse_key'):
    d = collapsed_data.copy()
    q = condition_df['Sample'].unique().tolist()
    meta = [c for c in d.columns if c not in q]
    qd, md = d[q].T, d[meta].T
    qd.columns = md.loc[key_col]
    qd['group'] = qd.index.map(dict(zip(condition_df['Sample'], condition_df['Condition'])))
    qd['sample'] = qd['group'] + '_' + (qd.groupby('group').cumcount() + 1).astype(str)
    qd['subject'] = qd['sample']
    return qd

def normalize_phospho_median(phospho_df, protein_df):
    common = set(phospho_df.index) & set(protein_df.index)
    ph = phospho_df.loc[phospho_df.index.isin(common)]; pr = protein_df.loc[protein_df.index.isin(common)]
    ex = lambda x: x.split('~')[0] if '~' in x else x.split('_')[0]
    p2 = {s: ex(s) for s in ph.columns}; med = pr.groupby(pr.index).median(); out = ph.copy(); ok = set()
    for cond in common:
        m = ph.index == cond
        for s in ph.columns:
            mm = [c for c in med.columns if p2[s] in c.split(';')]
            if mm and not pd.isna(med.loc[cond, mm[0]]):
                out.loc[m, s] = ph.loc[m, s] - med.loc[cond, mm[0]]; ok.add(s)
    print(f'normalize_phospho_median: {len(ok)}/{len(ph.columns)} sites matched to a protein')
    return out[list(ok)]

def permanova(D2, labels, nperm=999, seed=0):
    labels = np.asarray(labels); N = len(labels); a = len(np.unique(labels))
    def ssw(l):
        s = 0.0
        for g in np.unique(l):
            idx = np.where(l == g)[0]; ng = len(idx)
            s += D2[np.ix_(idx, idx)][np.triu_indices(ng, 1)].sum() / ng
        return s
    SST = D2[np.triu_indices(N, 1)].sum() / N; SSW = ssw(labels)
    F = ((SST - SSW) / (a - 1)) / (SSW / (N - a))
    rng = np.random.default_rng(seed)
    cnt = sum(1 for _ in range(nperm)
              if (lambda w: ((SST - w) / (a - 1)) / (w / (N - a)))(ssw(rng.permutation(labels))) >= F)
    return F, (SST - SSW) / SST, (cnt + 1) / (nperm + 1)

# Figure 5e
PCA of the neuron-population phosphoproteome (excitatory/inhibitory x cortex/subcortex), after sample QC, global >=70% valid-value filter, down-shifted imputation, and parent-protein normalization. *PCA separation is weak (low PC1/PC2 variance); separation is quantified by PERMANOVA (annotated) testing cell type and anatomical region directly — replaces the 'clear separation' wording (reviewer #75/#50).*

In [15]:
# Figure 5e (part 1) - neuron phospho preprocessing: QC -> condition -> global 0.7 filter -> impute.
sd = process_ptm_site_report(pd.read_csv(WHISPER, sep='\t', low_memory=False), cutoff=0.75)['site_data']
sample_all = [c for c in sd.columns if c not in PROC_META]
meta_cols  = [c for c in sd.columns if c in PROC_META]
counts = {c: int(sd[c].notna().sum()) for c in sample_all}
qc_thr = np.nanmedian(list(counts.values())) - stats.iqr(list(counts.values())) * 1.5
keep_samples = [c for c in sample_all if counts[c] >= qc_thr]
print(f'sample QC: {len(sample_all)} -> {len(keep_samples)} (threshold {qc_thr:.0f} sites)')
sd_qc = sd[keep_samples + meta_cols]

ph = pd.read_csv(PHOS_COND, sep='\t')
ph['Condition'] = ph['Condition'].str[:-1]                      # act_cortex1 -> act_cortex (4 groups)
ph['Sample'] = ph['Run Label'].str.split('.').str[0]           # strip .raw to match report columns
condition_phospho = ph[['Sample', 'Condition']]
condition_phospho = condition_phospho[condition_phospho['Sample'].isin(keep_samples)]
print('groups:', condition_phospho['Condition'].value_counts().to_dict())

df_cond = assign_condition_setup(sd_qc, condition_phospho)
df_filt = df_cond.loc[:, (1 - (df_cond.isna().sum() / len(df_cond))) >= 0.7]   # global >=70% (v00)
df_imp = ac.imputation_normal_distribution(df_filt).reset_index()
print(f'after global 0.7 filter + impute: {df_imp.shape[0]} samples x {df_imp.shape[1]-3} sites')

Dropped 1,782 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 95,325 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 9,784 → 9,722.
Final: 9,722 sites × 30 samples.
sample QC: 30 -> 26 (threshold 2232 sites)
groups: {'inh_cortex': 8, 'act_cortex': 6, 'act_subcortex': 6, 'inh_subcortex': 6}
after global 0.7 filter + impute: 26 samples x 2009 sites


In [16]:
# Figure 5e (part 2) - neuron proteome + parent-protein normalization.
prot = pd.read_csv(PROT_REPORT, sep='\t').set_index('PG.ProteinGroups')
pr = pd.read_csv(PROT_COND, sep='\t')
pr['Condition'] = pr['Condition'].str[:-2]                      # act_cortex_1 -> act_cortex
rl2c = dict(zip(pr['Run Label'], pr['Condition']))
def _runlabel(col): return re.sub(r'^\[\d+\] ', '', col).replace('.PG.Quantity', '')
keepc = [c for c in prot.columns if rl2c.get(_runlabel(c)) is not None]
prot = prot[keepc]; prot.columns = [rl2c[_runlabel(c)] for c in keepc]
prot_log = np.log2(prot)
print('proteome mapped:', len(keepc), 'cols ->', pd.Series(prot.columns).value_counts().to_dict())

collapsed2 = df_imp.drop(['sample', 'subject'], axis=1).set_index('group')
norm = normalize_phospho_median(collapsed2, prot_log.T).reset_index()
norm['sample'] = df_imp['sample'].values
norm['subject'] = df_imp['subject'].values
print(f'normalized: {norm.shape[0]} samples x {norm.shape[1]-3} sites')

proteome mapped: 35 cols -> {'inh_cortex': 12, 'inh_subcortex': 11, 'act_cortex': 9, 'act_subcortex': 3}
normalize_phospho_median: 1991/2009 sites matched to a protein
normalized: 26 samples x 1991 sites


In [17]:
# Figure 5e (part 3) - PCA + 95% data ellipses + PERMANOVA (overall / cell type / region).
pca = ac.run_pca(norm)
print('explained variance:', pca[1])
pdf = pca[0][0]

feat = [c for c in norm.columns if c not in ('group', 'sample', 'subject')]
D2 = squareform(pdist(norm[feat].values, 'euclidean')) ** 2
grp = norm['group'].values
celltype = np.array(['act' if g.startswith('act') else 'inh' for g in grp])
region   = np.array(['subcortex' if 'subcortex' in g else 'cortex' for g in grp])
res = {}
for name, lab in [('overall (4 groups)', grp), ('cell type (act/inh)', celltype), ('region (ctx/subctx)', region)]:
    F, R2, pv = permanova(D2, lab)
    res[name] = (F, R2, pv)
    print(f'PERMANOVA {name:<22} pseudo-F={F:.2f}  R2={R2:.3f}  p={pv:.4f}')

CMAP = {'act_cortex': '#2a9d8f', 'act_subcortex': '#f4a261', 'inh_cortex': '#264653', 'inh_subcortex': '#e76f51'}

def _ellipse(xs, ys, chi2_95=5.991, npts=120):
    cov = np.cov(xs, ys); mean = np.array([np.mean(xs), np.mean(ys)])
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2 * np.pi, npts); circ = np.array([np.cos(t), np.sin(t)])
    ell = mean[:, None] + vecs @ (np.sqrt(chi2_95) * np.sqrt(vals)[:, None] * circ)
    return ell[0], ell[1]

fig = px.scatter(pdf, x='x', y='y', color='group', hover_name='sample', color_discrete_map=CMAP)
fig.update_traces(marker=dict(size=21, line=dict(width=1, color='black')))
for g, c in CMAP.items():
    sub = pdf[pdf['group'] == g]
    if len(sub) >= 3:
        ex, ey = _ellipse(sub['x'].values, sub['y'].values)
        fig.add_trace(go.Scatter(x=ex, y=ey, mode='lines', line=dict(color=c, width=2),
                                 fill='toself', fillcolor=_hex_to_rgba(c, 0.08),
                                 showlegend=False, hoverinfo='skip'))
ct = res['cell type (act/inh)']; rg = res['region (ctx/subctx)']
ann = ('PERMANOVA<br>cell type: p=%.3f (R2=%.2f)<br>region: p=%.3f (R2=%.2f)'
       % (ct[2], ct[1], rg[2], rg[1]))
fig.add_annotation(x=0.02, y=0.98, xref='paper', yref='paper', showarrow=False, align='left', text=ann)
fig.update_layout(width=680, height=600, template='plotly_white',
                  xaxis_title=pca[1]['x_title'], yaxis_title=pca[1]['y_title'], showlegend = False)
fig.show()
fig.write_image(r'figures/figure5/figure5e.pdf', width=600, height=600)

explained variance: {'x_title': 'PC1 (0.11)', 'y_title': 'PC2 (0.10)', 'group': 'group'}
PERMANOVA overall (4 groups)     pseudo-F=1.69  R2=0.187  p=0.0010
PERMANOVA cell type (act/inh)    pseudo-F=1.66  R2=0.065  p=0.0010
PERMANOVA region (ctx/subctx)    pseudo-F=1.90  R2=0.073  p=0.0010


# Figure 5f
ANOVA across the four neuron populations (one-way, BH-FDR<0.05) on the parent-protein-normalized phosphoproteome, followed by hierarchical clustering of the significant sites (z-scored group means), per-cluster intensity trajectories, and per-cluster GO/pathway enrichment. Panels are shown full-size (reviewer #76).

In [18]:
# Figure 5f helper functions (clustering + GO with proteome background), ported from v00.
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from Bio import SeqIO
import gseapy as gp

def z_normalize_data(df):
    return pd.DataFrame(StandardScaler().fit_transform(df.T).T, index=df.index, columns=df.columns)

def perform_hierarchical_clustering(df_norm, method='ward', metric='euclidean'):
    return (linkage(df_norm.T, method=method, metric=metric),
            linkage(df_norm,   method=method, metric=metric))

def extract_clusters(linkage_matrix, labels, n_clusters=4):
    clusters = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    return pd.DataFrame({'Item': labels, 'Cluster': clusters})

def plot_clustermap(df_norm, figsize=(5, 12), cmap='rocket', method='ward', metric='euclidean',
                    n_clusters=4, save_path=None):
    ls, lf = perform_hierarchical_clustering(df_norm, method=method, metric=metric)
    fclust = fcluster(lf, n_clusters, criterion='maxclust')
    colors = plt.cm.Set3(np.linspace(0, 1, n_clusters))
    row_colors = [colors[c - 1] for c in fclust]
    g = sns.clustermap(df_norm, method=method, metric=metric, cmap=cmap, center=0, figsize=figsize,
                       cbar_kws={'label': 'Z-score'}, xticklabels=True,
                       yticklabels=False if df_norm.shape[0] > 50 else True,
                       dendrogram_ratio=0.15, colors_ratio=0.03, row_colors=row_colors)
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return g

def uniprot_to_gene_from_fasta(fasta_path):
    mapping = {}
    for record in SeqIO.parse(fasta_path, 'fasta'):
        header = record.description
        parts = header.split('|')
        uid = parts[1] if len(parts) >= 2 else header.split()[0]
        gn = next((f[3:] for f in header.split() if f.startswith('GN=')), None)
        if gn:
            mapping[uid] = gn
    return mapping

def run_go_enrichment_per_cluster(cluster_gene_dict, background, gene_sets, cutoff=0.05):
    """ORA per cluster with a custom (dataset proteome) background, via gp.enrich (same as Fig 4k/l)."""
    out = {}
    for name, genes in cluster_gene_dict.items():
        genes = [g for g in genes if isinstance(g, str)]
        if len(genes) < 3:
            print(f'skip {name}: {len(genes)} genes'); continue
        try:
            enr = gp.enrich(gene_list=genes, gene_sets=list(gene_sets),
                            background=background, outdir=None, cutoff=cutoff)
            out[name] = {'results': enr.results}
            nsig = (enr.results['Adjusted P-value'] < 0.05).sum()
            print(f'{name}: {len(genes)} genes vs {len(background)} bg -> {nsig} sig terms')
        except Exception as e:
            print(f'{name}: gp.enrich failed ({e})')
    return out

In [19]:
# Figure 5f (part 1) - ANOVA -> significant sites -> z-scored heatmap (PLOTLY, rocket; ALL 4 clusters).
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram
CUSTOM_ORDER = ['act_subcortex', 'inh_subcortex', 'act_cortex', 'inh_cortex']
CLUSTER_ORDER = [1, 2, 3, 4]          # display order in the heatmap (all clusters)
FOCUS_CLUSTERS = [1, 3]               # clusters carried to trajectories + GO (cortex vs subcortex)

anova = ac.run_anova(norm)                                    # one-way ANOVA on normalized data (4 groups)
sig_ids = set(anova.loc[anova['rejected'] == True, 'identifier'])
print(f'ANOVA significant sites (BH<0.05): {len(sig_ids)} of {norm.shape[1]-3}')

# group-mean matrix of the IMPUTED (pre-normalization) values, significant sites only (as in v00)
gm = df_imp.groupby('group').mean(numeric_only=True).T
gm.index.name = 'PTM_Collapse_key'
gm = gm[gm.index.isin(sig_ids)][CUSTOM_ORDER]
df_norm_hm = z_normalize_data(gm)

# 4-cluster assignment (same numbering as the seaborn render)
ls, lf = perform_hierarchical_clustering(df_norm_hm)
feature_clusters = extract_clusters(lf, df_norm_hm.index.tolist(), n_clusters=4)
feature_clusters['gene_symbol'] = feature_clusters['Item'].apply(lambda x: x.split('~')[1].split('_')[0])
feature_clusters.columns = ['PTM_Collapse_key', 'Cluster', 'gene_symbol']
print('cluster sizes:', feature_clusters['Cluster'].value_counts().to_dict())

# rocket colorscale -> plotly
_rk = sns.color_palette('rocket', as_cmap=True)
ROCKET = [[i/31, 'rgb(%d,%d,%d)' % tuple(int(255*c) for c in _rk(i/31)[:3])] for i in range(32)]

# rows: all clusters, grouped in CLUSTER_ORDER, leaf order within each cluster
clmap = dict(zip(feature_clusters['PTM_Collapse_key'], feature_clusters['Cluster']))
rows = df_norm_hm.index.tolist()
leaf_order = [rows[i] for i in dendrogram(lf, no_plot=True)['leaves']]
keep = [r for cl in CLUSTER_ORDER for r in leaf_order if clmap[r] == cl]
mat = df_norm_hm.loc[keep, CUSTOM_ORDER]
print('heatmap rows (all clusters):', len(mat))



d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Pro

ANOVA significant sites (BH<0.05): 246 of 1991
cluster sizes: {3: 82, 4: 62, 1: 58, 2: 44}
heatmap rows (all clusters): 246


d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Pro

In [20]:
# Figure 5f (part 2) - z-scored intensity trajectories for the focus clusters (1 & 3).
dfc = df_norm_hm.reset_index().merge(feature_clusters[['PTM_Collapse_key', 'Cluster']],
                                     on='PTM_Collapse_key').set_index('PTM_Collapse_key')
for cl in FOCUS_CLUSTERS:
    sub = dfc[dfc['Cluster'] == cl][CUSTOM_ORDER]
    fig = go.Figure()
    for _, row in sub.iterrows():
        fig.add_trace(go.Scatter(x=CUSTOM_ORDER, y=row.values, mode='lines',
                                 line=dict(color='lightgrey', width=0.4), opacity=0.5, showlegend=False))
    fig.add_trace(go.Scatter(x=CUSTOM_ORDER, y=sub.mean().values, mode='lines+markers',
                             line=dict(color='#b3321e', width=2.5), marker=dict(size=8), showlegend=False))
    fig.update_layout(width=600, height=420, template='plotly_white',
                      title=f'Cluster {cl} (n={len(sub)})', yaxis_title='Z-score')
    fig.show()
    fig.write_image(rf'figures/figure5/figure5f_cluster{cl}.pdf', width=600, height=420)

# Figure 5f (GO enrichment)
Per-cluster GO/pathway enrichment (Enrichr, Mouse). The full significant-term lists are printed; the bar panels show neuronal/synaptic-relevant terms (keyword-selected, top by Combined Score) — revise the keyword list as needed.

In [21]:
# Figure 5f (part 3) - per-cluster GO enrichment (clusters 1 & 3), proteome background. Needs internet.
MOUSE_FASTA = r'pride_data/analysis_data/figure5/mouse.fasta'
fasta_map = uniprot_to_gene_from_fasta(MOUSE_FASTA)
prot_acc = pd.read_csv(PROT_REPORT, sep='\t')['PG.ProteinGroups']
background_genes = sorted({fasta_map[u.strip()] for pg in prot_acc
                           for u in str(pg).split(';') if u.strip() in fasta_map})
print(f'background = detected proteome: {len(background_genes)} genes')

cluster_gene_dict = {f'Cluster_{c}': feature_clusters[feature_clusters['Cluster'] == c]['gene_symbol'].dropna().unique().tolist()
                     for c in FOCUS_CLUSTERS}
enrichment_results = run_go_enrichment_per_cluster(
    cluster_gene_dict, background=background_genes,
    gene_sets=['WikiPathways_2024_Mouse', 'GO_Biological_Process_2025', 'GO_Cellular_Component_2025',
               'GO_Molecular_Function_2025', 'KEGG_2019_Mouse', 'SynGO_2024'], cutoff=0.05)
for name, r in enrichment_results.items():
    sig = r['results'][r['results']['Adjusted P-value'] < 0.05]
    print(f'\n--- {name}: {len(sig)} sig terms ---')
    print(list(sig.sort_values('Combined Score', ascending=False)['Term'])[:20])

background = detected proteome: 7700 genes
Cluster_1: 47 genes vs 7700 bg -> 5 sig terms
Cluster_3: 70 genes vs 7700 bg -> 59 sig terms

--- Cluster_1: 5 sig terms ---
['Anchored Component Of Postsynaptic Density Membrane (GO:0099031) CC', 'Structural Constituent Of Postsynaptic Density (GO:0098919) BP', 'Neurotransmitter Receptor Localization To Postsynaptic Specialization Membrane (GO:0099645) BP', 'Splicing Factor NOVA Regulated Synaptic Proteins WP1983', 'Postsynapse (GO:0098794) CC']

--- Cluster_3: 59 sig terms ---
['Protein Kinase A Catalytic Subunit Binding (GO:0034236)', 'Synaptic Vesicle Cycle (GO:0099504) BP', 'Regulation of Endoplasmic Reticulum Tubular Network Organization (GO:1903371)', 'Positive Regulation of Heart Rate (GO:0010460)', 'Filamin Binding (GO:0031005)', 'Pyrimidine Nucleobase Catabolic Process (GO:0006208)', 'Synaptic Vesicle Localization (GO:0097479)', 'cAMP-dependent Protein Kinase Inhibitor Activity (GO:0004862)', 'Regulation of Protein Lipidation (GO:190

In [22]:
# Figure 5f (part 4) - per-cluster GO bar plots (neuronal/synaptic keyword selection, top by Combined Score).
PURPLE = [[0, '#5a4a6f'], [0.25, '#9970ab'], [0.5, '#c994c7'], [0.75, '#d4b9da'], [1, '#e0d0e8']]
NEURO_KW = ['SYNAP', 'NEURON', 'AXON', 'DENDRIT', 'VESICLE', 'CALCIUM', 'CALMODULIN', 'GLUTAMAT',
            'GABA', 'ION CHANNEL', 'POSTSYNAP', 'PRESYNAP', 'ACTIVE ZONE', 'CHROMATIN', 'SUMOYLATION',
            'TRANSLATION', 'PLASTICITY', 'CYTOSKELET', 'MICROTUBULE']

def go_bar(cluster_name, n=6):
    if cluster_name not in enrichment_results:
        print(f'{cluster_name}: no results'); return
    df = enrichment_results[cluster_name]['results']
    df = df[df['Adjusted P-value'] < 0.05].copy()
    df['_u'] = df['Term'].str.upper()
    sel = df[df['_u'].apply(lambda t: any(k in t for k in NEURO_KW))]
    sel = sel.sort_values('Combined Score', ascending=False).head(n).sort_values('Combined Score')
    print(f'{cluster_name}: {len(sel)} neuronal terms shown')
    if sel.empty:
        return
    fig = px.bar(sel, y='Term', x=np.log2(sel['Combined Score']), color='Adjusted P-value',
                 orientation='h', color_continuous_scale=PURPLE)
    fig.update_layout(width=1200, height=500, template='plotly_white', title=cluster_name,
                      xaxis_title='log2(Combined Score)', yaxis_title='')
    fig.show()
    fig.write_image(rf'figures/figure5/figure5f_{cluster_name}_GO.pdf', width=1200, height=500)

for name in enrichment_results:
    go_bar(name, n=6)

Cluster_1: 5 neuronal terms shown


Cluster_3: 6 neuronal terms shown


In [23]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
FIG = 5   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
_META={"Protein_group","Gene_group","PTM_0_aa","PTM_pos","PTM_mult123","PTM_flank","PTM_Collapse_key","PTM_localization","UPD_seq"}
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")
def _depth(dct, key):
    _r=[]
    for k in sorted(dct):
        for rep,(samp,c) in enumerate(count_sites_per_sample_ptm_report(dct[k]).items(),1):
            _r.append({"Raw file":samp, key:k, "Replicate":rep, "Number of class I sites":int(c)})
    return pd.DataFrame(_r)

_try(lambda: dump_panel(_depth(shapes,"Condition_shapes"), f"Figure {FIG}b"), f"Figure {FIG}b")
_try(lambda: dump_panel(corr_df, f"Figure {FIG}c"), f"Figure {FIG}c")
_try(lambda: dump_panel(pdf.rename(columns={"x":"PC1","y":"PC2"}), f"Figure {FIG}e"), f"Figure {FIG}e")
def _5f():
    m=df_norm_hm.reset_index().rename(columns={"index":"PTM_Collapse_key"})
    cl=dict(zip(feature_clusters["PTM_Collapse_key"], feature_clusters["Cluster"]))
    m["Cluster"]=m["PTM_Collapse_key"].map(cl)
    dump_panel(m, f"Figure {FIG}f")
_try(_5f,f"Figure {FIG}f")
def _5d():
    SITES=['P20357~Map2_S1783_M1','P10637~Mapt_S494_M1','P28652~Camk2b_T287_M1','P63318~Prkcg_T514_M1']
    _sd={n:process_ptm_site_report(shapes[n],cutoff=0.75)['site_data'] for n in SHAPE_ORDER}
    rows=[]
    for key in SITES:
        for n in SHAPE_ORDER:
            sd=_sd[n]; row=sd[sd['PTM_Collapse_key']==key]
            if row.empty: continue
            cols=[c for c in sd.columns if c not in PROC_META]
            for rep,v in enumerate([x for x in row[cols].iloc[0].values if pd.notna(x)],1):
                rows.append({'Site':key.split('~')[1],'PTM_Collapse_key':key,
                             'Condition':f'{n}shapes','Replicate':rep,'log2 intensity':float(v)})
    dump_panel(pd.DataFrame(rows), f"Figure {FIG}d")
_try(_5d,f"Figure {FIG}d")
print(f"Figure {FIG} export done.")


  [MetaInfo] wrote 'Figure 5b'  (24 rows x 4 cols)
  [MetaInfo] wrote 'Figure 5c'  (1243 rows x 7 cols)
  [MetaInfo] wrote 'Figure 5e'  (26 rows x 4 cols)
  [MetaInfo] wrote 'Figure 5f'  (246 rows x 6 cols)
Dropped 31 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 1,002 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 406 → 401.
Final: 401 sites × 4 samples.
Dropped 261 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 2,789 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,120 → 1,112.
Final: 1,112 sites × 4 samples.
Dropped 232 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 3,910 intensity cells masked (0.0%).
Deduplicated multi-protein rows